# Modelado de Fatiga con Random Forest Regressor

Este notebook contiene la explicación teórica, la revisión de literatura científica, y la implementación paso a paso de un **Random Forest Regressor** para predecir los niveles continuos de fatiga física y mental del dataset **FatigueSet**.

---

## 1. Fundamentos Teóricos y Literatura de Referencia

El algoritmo de **Random Forest** (Bosques Aleatorios), propuesto originalmente por Leo Breiman en 2001, es una técnica de aprendizaje supervisado ensemble basada en embolsado (*bagging*) y selección aleatoria de características. En tareas de regresión para series de tiempo, destaca por su robustez frente al ruido y su capacidad para modelar relaciones no lineales complejas sin sobreajustarse fácilmente.

### Principales Conceptos de Operación:
1.  **Bagging (Bootstrap Aggregating):** Se entrenan múltiples árboles de decisión ($B$) de forma independiente. Cada árbol se construye a partir de una muestra aleatoria de tamaño $N$ tomada con reemplazo (bootstrap) del conjunto de entrenamiento original. Esto ayuda a reducir la varianza global del estimador al promediar modelos independientes.
2.  **Subespacio Aleatorio (Feature Subspace Projection):** En lugar de evaluar todas las características para decidir el mejor punto de división de un nodo en cada árbol, el algoritmo restringe la búsqueda a un subconjunto aleatorio de variables de tamaño $m \approx p/3$ (donde $p$ es el número de características totales). Esto descorrelaciona los árboles entre sí, haciendo al bosque altamente resiliente al ruido fisiológico.
3.  **Agregación en Regresión:** La predicción final $\hat{f}(x)$ de un Random Forest Regressor para una nueva muestra $x$ se define como el promedio aritmético simple de las predicciones de los $B$ árboles de regresión individuales:
    $$\hat{f}(x) = \frac{1}{B} \sum_{b=1}^{B} T_b(x)$$
    Donde $T_b(x)$ es la predicción del $b$-ésimo árbol de decisión.

### Diagrama de Flujo del Proceso (Mermaid):
```mermaid
graph TD
    A["Dataset Original (N muestras, p features)"] --> B1["Muestra Bootstrap 1 (Con reemplazo)"]
    A --> B2["Muestra Bootstrap 2 (Con reemplazo)"]
    A --> B3["Muestra Bootstrap B (Con reemplazo)"]
    
    B1 --> C1["Entrenar Arbol 1 (Feature Selection m < p)"]
    B2 --> C2["Entrenar Arbol 2 (Feature Selection m < p)"]
    B3 --> C3["Entrenar Arbol B (Feature Selection m < p)"]
    
    C1 --> T1["Arbol Ajustado T_1"]
    C2 --> T2["Arbol Ajustado T_2"]
    C3 --> TB["Arbol Ajustado T_B"]
    
    D["Nueva Entrada Fisiologica (x)"] --> T1
    D --> T2
    D --> TB
    
    T1 --> E1["Prediccion y_1 = T_1(x)"]
    T2 --> E2["Prediccion y_2 = T_2(x)"]
    TB --> EB["Prediccion y_B = T_B(x)"]
    
    E1 --> F["Agregacion: Promedio de Predicciones"]
    E2 --> F
    EB --> F
    
    F --> G["Prediccion Final: f_hat(x) = (1/B) * sum(y_b)"]
```

### Referencias Científicas:
- **Breiman, L. (2001).** *"Random Forests"*. Machine Learning, 45(1), 5-32.
  [Enlace al Paper (Springer)](https://doi.org/10.1023/A:1010933404324)
- **Kane, M. J. et al. (2014).** *"Comparison of ARIMA and Random Forest time series models for prediction of avian influenza H5N1 outbreaks"*. BMC Bioinformatics.
  [Enlace al Paper (BioMed Central)](https://doi.org/10.1186/1471-2105-15-276)
  *(Demuestra que los modelos basados en Random Forest superan a los métodos estadísticos lineales clásicos como ARIMA en la modelización de series temporales no lineales e irregulares).*

In [1]:
# SETUP e IMPORTACIONES
import sys
import time
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Añadir fatigueset-lib al sys.path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset import FatigueSetPipeline
print("✓ Imports completados y path configurado.")

✓ Imports completados y path configurado.


## 2. Carga y Saneamiento del Dataset

Utilizamos la tubería del paquete `fatigueset` para cargar los datos crudos y generar el conjunto de datos normalizado.

In [2]:
dataset_path = str(Path.cwd().parent / "fatigueset")
pipeline = FatigueSetPipeline(dataset_path=dataset_path, umbral_nulos=5.0)

print("Cargando dataset a través del pipeline...")
data_res = pipeline.ejecutar(verbose=False, incluir_ventanas=False, normalizar=True)
df_ml = data_res['ml_normalizado']
print(f"✓ Dataset cargado correctamente: {df_ml.shape} (muestras x columnas)")

Cargando dataset a través del pipeline...


✓ Dataset cargado correctamente: (108, 31) (muestras x columnas)


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


## 3. Preparación de Características y Targets

Separamos las columnas de identidad (participante, sesión, etc.) de las variables predictivas y limpiamos valores no finitos (inf, NaN) para evitar que rompan el entrenamiento del modelo de Scikit-Learn.

In [3]:
targets = ['fatiga_fisica', 'fatiga_mental']
identity_cols = ['participante', 'sesion', 'intensidad', 'intensidad_num', 'fase', 'fase_num']
exclude_cols = identity_cols + targets
feature_cols = [c for c in df_ml.columns if c not in exclude_cols]

# Características (X)
X = df_ml[feature_cols].select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).fillna(0)
print(f"Dimensiones de X: {X.shape}")
print(f"Características predictivas: {list(X.columns)}")

Dimensiones de X: (108, 23)
Características predictivas: ['hr_media', 'hr_std', 'br_media', 'br_std', 'hrv_media', 'hrv_std', 'eda_media', 'eda_std', 'eeg_alpha_media', 'eeg_beta_media', 'eeg_theta_media', 'delta_fisica_ejercicio', 'delta_mental_ejercicio', 'delta_fisica_cognitivo', 'delta_mental_cognitivo', 'delta_fisica_total', 'delta_mental_total', 'fisica_M1', 'fisica_M2', 'fisica_M3', 'mental_M1', 'mental_M2', 'mental_M3']


## 4. Entrenamiento y Evaluación Paso a Paso

Ajustamos un modelo de Random Forest para predecir la **Fatiga Mental**, evaluándolo con validación cruzada y guardando el estimador resultante.

In [4]:
target_var = 'fatiga_mental'
y = df_ml[target_var].replace([np.inf, -np.inf], np.nan).fillna(0)

# Parámetros de Random Forest
seed = 42
n_estimators = 100
max_depth = 8

print(f"Entrenando estimador para {target_var} con {n_estimators} árboles (profundidad máxima={max_depth})...")
rf_model = RandomForestRegressor(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=seed,
    n_jobs=-1
)

# 1. Validación Cruzada (R²)
kfold = KFold(n_splits=5, shuffle=True, random_state=seed)
t_cv_start = time.time()
cv_r2_scores = cross_val_score(rf_model, X, y, cv=kfold, scoring='r2')
time_cv = time.time() - t_cv_start

# 2. Ajuste completo y toma de tiempos
t_fit_start = time.time()
rf_model.fit(X, y)
time_fit = time.time() - t_fit_start

# 3. Predicciones y cálculo de métricas de regresión
y_pred = rf_model.predict(X)
mae = mean_absolute_error(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y, y_pred)

# 4. Contador de parámetros (cantidad total de nodos de decisión en el bosque)
total_nodes = sum(tree.tree_.node_count for tree in rf_model.estimators_)

print(f"\n✓ Validación Cruzada 5-Fold R²: {np.mean(cv_r2_scores):.4f} (± {np.std(cv_r2_scores):.4f})")
print(f"✓ Métricas de Ajuste General (Entrenamiento completo):")
print(f"  R²: {r2:.4f}")
print(f"  MAE: {mae:.4f} puntos")
print(f"  RMSE: {rmse:.4f} puntos")
print(f"  Tiempo de Entrenamiento del Bosque: {time_fit:.4f}s (Total CV: {time_cv:.2f}s)")
print(f"  Número total de nodos en el bosque (Parámetros): {total_nodes}")

Entrenando estimador para fatiga_mental con 100 árboles (profundidad máxima=8)...



✓ Validación Cruzada 5-Fold R²: 0.0810 (± 0.1522)
✓ Métricas de Ajuste General (Entrenamiento completo):
  R²: 0.8743
  MAE: 5.6881 puntos
  RMSE: 6.9940 puntos
  Tiempo de Entrenamiento del Bosque: 0.1908s (Total CV: 1.29s)
  Número total de nodos en el bosque (Parámetros): 9178


## 5. Serialización del Modelo

Persistimos el modelo entrenado en disco dentro de la carpeta centralizada `/models/classicos/`.

In [5]:
output_dir = Path.cwd().parent / "models" / "classicos"
output_dir.mkdir(parents=True, exist_ok=True)

file_path = output_dir / "random_forest_fatiga_mental_notebook.pkl"
with open(file_path, 'wb') as f:
    pickle.dump(rf_model, f)

print(f"✓ Modelo guardado exitosamente en: {file_path}")

✓ Modelo guardado exitosamente en: C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\models\classicos\random_forest_fatiga_mental_notebook.pkl
